# EFSM Phase 1 — Model Verification

**Goal:** Confirm that `Qwen2.5-Omni-7B-Instruct` loads correctly on Kaggle T4 GPU in 4-bit, runs text inference, and produces speech output.

**Expected outcomes:**
- Model loads without CUDA OOM (~8–10 GB VRAM used)
- Text chat inference returns an empathetic response
- Speech generation saves a `.wav` file

---
**Before running:** Enable GPU T4 x2 in Kaggle Session Options, and add `HF_TOKEN` + `WANDB_API_KEY` as Kaggle Secrets.

## Cell 1 — Load Kaggle Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')

print('Secrets loaded.')

## Cell 2 — Clone Repo and Install Dependencies

In [ ]:
# Replace YOUR_USERNAME with your GitHub username
!git clone https://github.com/YOUR_USERNAME/empathetic-voice-llm.git
%cd empathetic-voice-llm
!pip install -r requirements.txt -q

## Cell 3 — Load Qwen2.5-Omni-7B-Instruct in 4-bit

In [ ]:
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from transformers import BitsAndBytesConfig
from huggingface_hub import login

# Explicitly log in so all HF downloads pick up the token automatically
login(token=os.environ["HF_TOKEN"])

MODEL_ID = "Qwen/Qwen2.5-Omni-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"VRAM before loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"],
)

processor = Qwen2_5OmniProcessor.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"],
)

print(f"VRAM after loading:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Model loaded successfully.")

## Cell 4 — Parameter Counts

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params / 1e9:.2f} B")
print(f"Trainable parameters: {trainable_params / 1e6:.2f} M  ({100 * trainable_params / total_params:.2f}%)")

## Cell 5 — VRAM Summary

In [ ]:
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved  = torch.cuda.memory_reserved(i)  / 1e9
    total_mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i} ({torch.cuda.get_device_name(i)}):")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total_mem:.2f} GB")

## Cell 6 — Text-mode Chat Inference

Sends a short empathetic prompt in text-only mode (no audio input). This confirms the Thinker is working correctly.

In [ ]:
SYSTEM_PROMPT = (
    "You are an empathetic voice therapist. Your role is to listen carefully "
    "to how the person feels, acknowledge their emotions, and respond with genuine "
    "warmth and understanding. Validate their feelings. Do not offer unsolicited "
    "advice. Make them feel truly heard."
)

USER_MESSAGE = "I feel so overwhelmed lately. Everything at uni feels like too much and I don't know how to cope."

messages = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": USER_MESSAGE},
]

# Use text-only mode: pass audio=None to avoid requiring an audio file
text_input = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(
    text=text_input,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

# Decode only the newly generated tokens
generated = output_ids[:, inputs["input_ids"].shape[-1]:]
response_text = processor.decode(generated[0], skip_special_tokens=True)

print("=" * 60)
print("USER:")
print(USER_MESSAGE)
print()
print("MODEL RESPONSE:")
print(response_text)
print("=" * 60)

## Cell 7 — Speech Generation (Talker Verification)

Generates a spoken audio response using the Talker module. Saves the output as `verify_output.wav`.

In [ ]:
import soundfile as sf
import numpy as np

SHORT_TEXT = "I hear you, and I want you to know that feeling overwhelmed is completely valid."

tts_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "Please say the following out loud."},
    {"role": "assistant", "content": SHORT_TEXT},
]

tts_text = processor.apply_chat_template(
    tts_messages,
    tokenize=False,
    add_generation_prompt=False,
)

tts_inputs = processor(
    text=tts_text,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    # generate() with return_audio=True activates the Talker + Token2Wav path
    audio_output = model.generate(
        **tts_inputs,
        max_new_tokens=512,
        return_audio=True,
    )

# audio_output is a tensor of shape [1, num_samples] at 24kHz
if hasattr(audio_output, 'audio') and audio_output.audio is not None:
    waveform = audio_output.audio[0].float().cpu().numpy()
elif isinstance(audio_output, torch.Tensor):
    waveform = audio_output[0].float().cpu().numpy()
else:
    # Fallback: try to get audio from the last element of output tuple
    waveform = audio_output[-1][0].float().cpu().numpy() if isinstance(audio_output, (list, tuple)) else None

OUTPUT_PATH = "verify_output.wav"
if waveform is not None:
    sf.write(OUTPUT_PATH, waveform, samplerate=24000)
    file_size = os.path.getsize(OUTPUT_PATH)
    print(f"Audio saved to {OUTPUT_PATH}")
    print(f"File size: {file_size / 1024:.1f} KB")
    print(f"Duration:  {len(waveform) / 24000:.2f} seconds")
    print()
    print("Phase 1 COMPLETE: model loads, text inference works, speech generation works.")
else:
    print("WARNING: Could not extract audio waveform from model output.")
    print("Text inference still succeeded. Check model's return_audio API.")

## Summary

If all cells ran without error:

| Check | Expected |
|-------|----------|
| VRAM after loading | ~8–10 GB (4-bit) |
| Text response | Empathetic reply printed above |
| `verify_output.wav` | Exists, > 0 KB |

**Phase 1 is complete.** Report any CUDA OOM or import errors back for debugging.